# nb03 — Explorador de células & Rosenfeld (saídas do script 13 v2)

Carrega `curva_reT_<onda>.csv`, `celulas_tabela_<onda>.csv` e as `fig_celulas_*.png`.
Escolha a onda e a célula para ver o r_e(T) **frente × traseira**, e navegue nas imagens.

In [ ]:
import os, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
import ipywidgets as W; from IPython.display import display, Image
import importlib, ondas_config; importlib.reload(ondas_config)
from ondas_config import ONDAS
BASE='resultados/rosenfeld_celulas'
ondas_disp=[o for o in ONDAS if os.path.isdir(f'{BASE}/{o}')]
print('ondas com saída do 13:',ondas_disp)

## r_e(T) por célula — frente × traseira

In [ ]:
EST={'frente':('#0072B2','o','-'),'traseira':('#D55E00','s','--'),'inteira':('#009E73','^','-')}
def curva(onda, celula):
    f=f'{BASE}/{onda}/curva_reT_{onda}.csv'
    if not os.path.exists(f): print('sem CSV para',onda); return
    d=pd.read_csv(f); d=d[(d.celula==celula)&(d.valido_re==1)]
    if d.empty: print('sem r_e válido (provável noite) para',celula); return
    plt.figure(figsize=(6,5))
    for lado,(cor,mk,ls) in EST.items():
        g=d[d.metade==lado]
        if g.empty: continue
        gg=g.groupby('T_bin').re_proxy_med.median().reset_index()
        plt.plot(gg.re_proxy_med,gg.T_bin,color=cor,marker=mk,ls=ls,label=lado)
    plt.gca().invert_yaxis(); plt.xlabel('r_e proxy (µm) [3,9µm]'); plt.ylabel('Tb topo (°C)')
    plt.title(f'{onda} — {celula}'); plt.legend(); plt.grid(alpha=.3); plt.show()

def atualiza_celulas(*_):
    onda=w_onda.value; f=f'{BASE}/{onda}/celulas_tabela_{onda}.csv'
    if os.path.exists(f):
        t=pd.read_csv(f); w_cel.options=sorted(t.celula.unique(), key=lambda x:int(x[1:]))
w_onda=W.Dropdown(options=ondas_disp, description='onda')
w_cel=W.Dropdown(options=[], description='célula')
w_onda.observe(atualiza_celulas,'value'); atualiza_celulas()
W.interact(curva, onda=w_onda, celula=w_cel);

## Navegar nas imagens das células (fig_celulas_*.png)

In [ ]:
def galeria(onda):
    arqs=sorted(glob.glob(f'{BASE}/{onda}/fig_celulas_*.png'))
    if not arqs: print('sem imagens para',onda); return
    def mostra(i=0): display(Image(filename=arqs[i]))
    W.interact(mostra, i=W.IntSlider(min=0,max=len(arqs)-1,value=0,description='quadro'))
W.interact(galeria, onda=W.Dropdown(options=ondas_disp, description='onda'));

## Tabela de células (núcleos, flashes, movimento)

In [ ]:
def tabela(onda):
    f=f'{BASE}/{onda}/celulas_tabela_{onda}.csv'
    if not os.path.exists(f): print('sem tabela'); return
    t=pd.read_csv(f)
    g=t.groupby('celula').agg(nflash_max=('nflash','max'),tbmin=('tbmin','min'),
        area_max=('area_nuc','max'),nquadros=('quando','nunique'),elet=('eletrificada','max')).reset_index()
    display(g.sort_values('nflash_max',ascending=False))
W.interact(tabela, onda=W.Dropdown(options=ondas_disp, description='onda'));